# 05장. 개인추천 점수 설계

| 핵심 질문 | 학습 시간 |
|---|---:|
| 개인마다 다른 추천 순위를 어떻게 만들까? | 4회차 전반 · 약 90분 |


## 이 장에서 배울 내용

- 추천 점수의 기준점·보너스·감점을 계산할 수 있다.
- 같은 급식도 가상 취향에 따라 순위가 달라지는 이유를 설명할 수 있다.
- 가상 알레르기 번호가 점수 계산 전에 제외되는 이유를 말할 수 있다.


## 생각 열기

같은 급식표를 보고도 어떤 학생은 면을, 어떤 학생은 밥과 국을 먼저 고릅니다. 추천기는 이 차이를 점수 규칙으로 나타냅니다. 어떤 조건이 점수를 올리고 내리는지 누구나 확인할 수 있어야 합니다.


## 핵심 용어

| 용어 | 뜻 |
|---|---|
| **기준점** | 보너스와 감점을 적용하기 전의 출발 점수 |
| **가중치** | 어떤 조건을 얼마나 크게 반영할지 정한 수 |
| **클리핑** | 결과를 정한 최소·최대 범위 안으로 자르는 일 |
| **설명 가능한 AI** | 결과가 나온 근거를 사용자가 확인할 수 있는 AI |


## 개념 익히기


추천 점수는 시험 성적이 아니라 정렬을 위한 상대 숫자입니다.

**20 + 70×텍스트 유사도 + 8×좋아함 일치 + 5×유형 일치 - 18×기피 일치 - 3×매운맛 차이**

마지막에는 0~100 사이로 자릅니다. 20점 기준점은 감점이 0점 아래로 바로 사라지지 않게 합니다. 가상 알레르기 번호는 점수를 낮추는 것이 아니라 후보에서 먼저 제외합니다.


## 활동 전 생각


유사도 0.2, 좋아함 1개, 면 유형 1개, 기피 0개, 매운맛 차이 1이라면<br>
20 + 14 + 8 + 5 - 0 - 3 = 44점입니다. 손으로 다시 계산해 보세요.


## 예상하기

- 파스타·피자·면을 좋아하는 가상 프로필의 1위 이유에 좋아하는 키워드가 표시된다.
- 추천 결과는 3행이다.


## 활동 1. 첫 가상 프로필 추천


### 코드 살펴보기


1. `PreferenceProfile`은 좋아함, 기피, 선호 유형, 매운맛, 가상 번호를 묶습니다.<br>
2. `recommend_menus`는 `meal_df`와 `profile`을 비교해 상위 세 행을 만듭니다.<br>
3. `reason` 열에는 각 점수가 만들어진 근거가 기록됩니다.


In [ ]:
import sys
from pathlib import Path

current_folder = Path.cwd().resolve()
for candidate in (current_folder, *current_folder.parents):
    if (candidate / "jupyter_course" / "notebook_support.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError(
        "프로젝트 폴더를 찾지 못했습니다. neis-meal-ai 폴더에서 "
        r".\.venv\Scripts\python.exe -m notebook 명령으로 다시 시작하세요."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jupyter_course.notebook_support import course_setup

setup = course_setup(PROJECT_ROOT)
PROJECT_ROOT = setup["root"]
raw_rows = setup["rows"]
meal_df = setup["frame"]
data_source = setup["source"]
print("프로젝트 폴더:", PROJECT_ROOT)
print("데이터 출처:", data_source)
print("급식 행 수:", len(raw_rows))

from neis_meal_ai.recommender import PreferenceProfile, recommend_menus

profile = PreferenceProfile(
    likes=("파스타", "피자"),
    avoids=("오이",),
    preferred_types=("면", "디저트"),
    spice_level=2,
    allergy_codes=(),
)
recommendation = recommend_menus(meal_df, profile, top_n=3)
print(recommendation[["date", "score", "menu_text", "reason"]].to_string(index=False))


### 결과 해석하기

reason 열에서 텍스트 유사도, 좋아하는 키워드, 유형, 기피, 매운맛 차이를 확인할 수 있습니다.


## 활동 2. 서로 다른 두 취향 비교


### 코드 살펴보기


1. `profile_b`에는 첫 프로필과 다른 밥·국 선호를 적습니다.<br>
2. `recommendation_b`는 같은 급식 표에 두 번째 취향을 적용한 결과입니다.<br>
3. 두 표의 `iloc[0]`을 비교하면 각 프로필의 1위가 보입니다.


In [ ]:
profile_b = PreferenceProfile(
    likes=("밥", "국"),
    avoids=(),
    preferred_types=("밥", "국물"),
    spice_level=3,
    allergy_codes=(),
)
recommendation_b = recommend_menus(meal_df, profile_b, top_n=3)
print("A의 1위:", recommendation.iloc[0]["menu_text"])
print("B의 1위:", recommendation_b.iloc[0]["menu_text"])

chapter_result = {
    "chapter": "05",
    "recommendations": len(recommendation),
    "top_score": float(recommendation.iloc[0]["score"]),
    "top_reason": str(recommendation.iloc[0]["reason"]),
}


### 결과 해석하기

데이터는 같아도 입력 취향이 달라지면 유사도와 보너스가 바뀌어 순위가 달라집니다.


## 탐구 활동

활동 1의 조건은 그대로 두고 `practice_like`만 ‘파스타’에서 ‘치킨’으로 바꾸어 1위와 추천 이유를 비교하세요. 실제 의료 정보는 적지 않습니다.

먼저 기본값으로 실행한 뒤 한 곳만 바꾸어 결과를 비교합니다.


In [ ]:
practice_like = "치킨"
practice_profile = PreferenceProfile(
    likes=(practice_like, "피자"),
    avoids=("오이",),
    preferred_types=("면", "디저트"),
    spice_level=2,
    allergy_codes=(),
)
practice_result = recommend_menus(meal_df, practice_profile, top_n=1)
print("기준 프로필 1위:", recommendation.iloc[0]["menu_text"])
print("좋아함만 바꾼 뒤:")
print(practice_result[["date", "score", "reason"]].to_string(index=False))


### 관찰 기록

- 바꾼 것:  
- 달라진 결과:  
- 그렇게 된 까닭:


## 확인 문제

1. 20점 기준점은 왜 있나요?
2. 기피 키워드는 점수에 어떻게 반영되나요?
3. 가상 알레르기 번호 일치 메뉴를 낮은 점수로 남기지 않고 제외하는 이유는 무엇인가요?


## 정답과 해설


1. 감점 효과가 0점 하한에서 바로 사라지지 않게 하기 위해서입니다.<br>
2. 한 번 일치할 때마다 18점을 뺍니다.<br>
3. 주의가 필요한 후보가 낮은 순위라도 추천 목록에 남지 않게 하기 위해서입니다. 그래도 실제 안전을 보장하지는 않습니다.


## 핵심 정리

- 추천 점수는 공개된 가감 규칙을 사용한다.
- 같은 데이터도 입력 취향에 따라 순위와 이유가 달라진다.
- 점수는 취향 비교용이며 건강·의학 점수가 아니다.

### 다음 장에서 배울 내용

06장에서는 추천 함수를 Jupyter 안에서 누를 수 있는 화면으로 연결합니다.


In [ ]:
import json
print("__CHAPTER_RESULT__=" + json.dumps(chapter_result, ensure_ascii=False))
